# Cell 1: Environment setup ONLY



In [ ]:
# ============================================================
# Cell 1: Environment setup ONLY
#   No data yet. Verify GPU + packages first.
# ============================================================

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---- Install ----
!pip install -q -U transformers datasets accelerate scikit-learn emoji

# ---- Core imports ----
import random, inspect, shutil, zipfile, itertools, re, html, gc, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
from scipy.special import softmax
import emoji

# ---- Reproducibility ----
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ---- Device ----
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    free_gb, total_gb = [x / 1e9 for x in torch.cuda.mem_get_info()]
    print(f"GPU free: {free_gb:.2f} / {total_gb:.2f} GB")

import transformers, sklearn
print(f"PyTorch: {torch.__version__} | Transformers: {transformers.__version__}")
print(f"\n✓ Cell 1 complete")

# Cell 2: Load all 6 files + apply cleaning



In [ ]:
# ============================================================
# Cell 2a: Check repo structure — find actual file names
# ============================================================
import os

# Check if repo exists
if os.path.exists("Hastika-ICON2026"):
    print("✓ Repo exists")
    print("\nRepo contents (top level):")
    for f in sorted(os.listdir("Hastika-ICON2026")):
        path = f"./Hastika-ICON2026/{f}"
        tag = "📁" if os.path.isdir(path) else "📄"
        print(f"  {tag} {f}")

    # Check data folder specifically
    if os.path.exists("Hastika-ICON2026/data"):
        print("\nData folder contents:")
        for f in sorted(os.listdir("Hastika-ICON2026/data")):
            size = os.path.getsize(f"./Hastika-ICON2026/data/{f}")
            print(f"  📄 {f}  ({size/1024:.1f} KB)")
    else:
        print("\n⚠️  No 'data' folder in repo")
        # Search for CSV files
        print("\nSearching for CSV files:")
        for root, dirs, files in os.walk("Hastika-ICON2026"):
            for f in files:
                if f.endswith(".csv"):
                    print(f"  {os.path.join(root, f)}")
else:
    print("✗ Repo does not exist")
    print("Will re-clone in next step")

In [ ]:
# ============================================================
# Cell 2a.2: Inspect each CSV's columns + sample rows
#   Understand which is train / validation / test
# ============================================================
import pandas as pd

files = [
    "binary_train.csv",
    "hastika_binary_validation.csv",
    "hastika_binary_test.csv",
    "multiclass_train.csv",
    "hastika_multiclass_validation.csv",
    "hastika_multiclass_test.csv",
]

DATA = "Hastika-ICON2026/data"

for fname in files:
    path = f"{DATA}/{fname}"
    df = pd.read_csv(path)
    print("=" * 70)
    print(f"📄 {fname}")
    print(f"   Shape: {df.shape}")
    print(f"   Columns: {list(df.columns)}")
    print(f"   Dtypes: {dict(df.dtypes)}")
    print(f"   First row:")
    for col in df.columns:
        val = str(df.iloc[0][col])[:80]
        print(f"      {col:20s}: {val}")
    print()

In [ ]:
# ============================================================
# Cell 2: Load all 6 files + apply cleaning
#   No split yet. Just load + clean + sanity check.
# ============================================================

# ---- Clone repo if needed ----
if not os.path.exists("Hastika-ICON2026"):
    !git clone -q https://github.com/shankarb14/Hastika-ICON2026.git
DATA = "Hastika-ICON2026/data"

# ---- Load all 6 CSVs ----
train_a  = pd.read_csv(f"{DATA}/binary_train.csv")
dev_a    = pd.read_csv(f"{DATA}/hastika_binary_validation.csv")
test_a   = pd.read_csv(f"{DATA}/hastika_binary_test.csv")

train_b  = pd.read_csv(f"{DATA}/multiclass_train.csv")
dev_b    = pd.read_csv(f"{DATA}/hastika_multiclass_validation.csv")
test_b   = pd.read_csv(f"{DATA}/hastika_multiclass_test.csv")

# ---- Preprocessing functions ----
def fix_mojibake(text):
    """Recover Kannada Unicode from UTF-8 bytes stored as Latin-1 chars."""
    if not isinstance(text, str):
        return text
    out, i = [], 0
    while i < len(text):
        if ord(text[i]) < 128:
            out.append(text[i]); i += 1; continue
        j = i
        while j < len(text) and ord(text[j]) >= 128:
            j += 1
        chunk = text[i:j]
        try:
            out.append(bytes(ord(c) for c in chunk if ord(c) < 256).decode("utf-8"))
        except (UnicodeDecodeError, ValueError):
            out.append(chunk)
        i = j
    return "".join(out)


def clean_text(text):
    """Mojibake fix + emoji→text + HTML/URL cleanup + lowercase."""
    if not isinstance(text, str):
        return ""
    text = fix_mojibake(text)
    text = emoji.demojize(text, delimiters=(" ", " "))
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", "@user", text)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text


# ---- Apply cleaning to all 6 dataframes ----
for df in [train_a, dev_a, test_a, train_b, dev_b, test_b]:
    df["Comment_clean"] = df["Comment"].apply(clean_text)

# ============================================================
# Sanity checks
# ============================================================
print("=" * 70)
print("Data loaded + cleaned")
print("=" * 70)

for name, df in [("train_a", train_a), ("dev_a", dev_a), ("test_a", test_a),
                 ("train_b", train_b), ("dev_b", dev_b), ("test_b", test_b)]:
    empty = (df["Comment_clean"].str.len() == 0).sum()
    print(f"  {name:8s}: {df.shape}  empty_after_cleaning={empty}")

# ---- ID overlap check (critical!) ----
print("\n" + "=" * 70)
print("ID overlap check")
print("=" * 70)

train_a_ids = set(train_a["id"])
dev_a_ids   = set(dev_a["id"])
test_a_ids  = set(test_a["id"])
print(f"  Task A train ∩ dev:  {len(train_a_ids & dev_a_ids)}  (must be 0)")
print(f"  Task A train ∩ test: {len(train_a_ids & test_a_ids)}  (must be 0)")
print(f"  Task A dev ∩ test:   {len(dev_a_ids & test_a_ids)}  (must be 0)")

train_b_ids = set(train_b["id"])
dev_b_ids   = set(dev_b["id"])
test_b_ids  = set(test_b["id"])
print(f"  Task B train ∩ dev:  {len(train_b_ids & dev_b_ids)}  (must be 0)")
print(f"  Task B train ∩ test: {len(train_b_ids & test_b_ids)}  (must be 0)")
print(f"  Task B dev ∩ test:   {len(dev_b_ids & test_b_ids)}  (must be 0)")

# ---- Label distribution ----
print("\n" + "=" * 70)
print("Label distributions")
print("=" * 70)
print(f"  train_a labels:\n{train_a['Label'].value_counts().to_string()}")
print(f"\n  dev_a labels:\n{dev_a['Label'].value_counts().to_string()}")
print(f"\n  train_b labels:\n{train_b['Hate Category'].value_counts().to_string()}")
print(f"\n  dev_b labels:\n{dev_b['Hate Category'].value_counts().to_string()}")

# ---- Sample cleaned comments ----
print("\n" + "=" * 70)
print("Sample cleaned comments")
print("=" * 70)
for i in range(3):
    row = train_a.iloc[i]
    print(f"  [{row['Label']}] {row['Comment_clean'][:100]}")

print(f"\n✓ Cell 2 complete")

# Cell 3: Labels, meta features, class weights


In [ ]:
# ============================================================
# Cell 3: Labels, meta features, class weights
# ============================================================

# ---- Missing imports ----
from sklearn.utils.class_weight import compute_class_weight

# ---- Labels (alphabetical = sklearn.classes_ order) ----
LABELS_A = ["Hate", "Non-Hate"]
L2ID_A   = {l: i for i, l in enumerate(LABELS_A)}
ID2L_A   = {i: l for l, i in L2ID_A.items()}

LABELS_B = ["Gender", "Geo-political", "Others", "Political", "Religion", "Violence"]
L2ID_B   = {l: i for i, l in enumerate(LABELS_B)}
ID2L_B   = {i: l for l, i in L2ID_B.items()}

# ---- HATE_LEXICON (define if not from Cell 1) ----
if "HATE_LEXICON" not in dir():
    HATE_LEXICON = {
        "bitch","slut","whore","rape","kill","die","bloody","bastard","fuck","fucker",
        "moron","idiot","stupid","asshole","dumb","shit","crap",
        "bhosdi","bhosad","bhosdike","madarchod","behenchod","randi","rundi","chutiya",
        "laude","lavde","gaand","gandu","harami","haraami","kutta","kamina","suar","tatte",
        "terrorist","bomb","violence","maar","mar","khoon","hatya",
    }

# ---- Meta features function ----
def meta_features(texts):
    """8 hand-crafted signals per comment."""
    feats = []
    for t in texts:
        t = str(t)
        n = max(len(t), 1)
        words = t.split()
        nw = max(len(words), 1)
        feats.append([
            len(t),
            sum(1 for c in t if c.isupper()) / n,
            t.count("!") + t.count("?"),
            len(re.findall(r":[a-z_]+:", t)),
            len(re.findall(r"(.)\1\1", t)),
            len(t) / nw,
            sum(1 for w in words if w in HATE_LEXICON),
            sum(1 for c in t if c.isdigit()) / n,
        ])
    return np.array(feats, dtype=float)

# ---- Meta features ----
META_TRAIN_A = meta_features(train_a["Comment_clean"].tolist())
META_DEV_A   = meta_features(dev_a["Comment_clean"].tolist())
META_TEST_A  = meta_features(test_a["Comment_clean"].tolist())

META_TRAIN_B = meta_features(train_b["Comment_clean"].tolist())
META_DEV_B   = meta_features(dev_b["Comment_clean"].tolist())
META_TEST_B  = meta_features(test_b["Comment_clean"].tolist())

# ---- Class weights ----
w_b = compute_class_weight(
    "balanced",
    classes=np.array(LABELS_B),
    y=train_b["Hate Category"].values,
)
W_B = torch.tensor(w_b, dtype=torch.float).to(DEVICE)

# ---- Integer labels ----
y_train_a_int = np.array([L2ID_A[x] for x in train_a["Label"].values])
y_dev_a_int   = np.array([L2ID_A[x] for x in dev_a["Label"].values])
y_train_b_int = np.array([L2ID_B[x] for x in train_b["Hate Category"].values])
y_dev_b_int   = np.array([L2ID_B[x] for x in dev_b["Hate Category"].values])

# ============================================================
# Summary
# ============================================================
print("=" * 70)
print("Cell 3 — Labels + meta features + class weights")
print("=" * 70)
print(f"Task A labels: {LABELS_A}")
print(f"Task B labels: {LABELS_B}")

print(f"\nMeta feature shapes:")
print(f"  TRAIN_A {META_TRAIN_A.shape}  DEV_A {META_DEV_A.shape}  TEST_A {META_TEST_A.shape}")
print(f"  TRAIN_B {META_TRAIN_B.shape}  DEV_B {META_DEV_B.shape}  TEST_B {META_TEST_B.shape}")

print(f"\nTask B class weights:")
for lbl, w in zip(LABELS_B, w_b):
    print(f"  {lbl:15s} {w:.3f}")

print(f"\nInteger labels:")
print(f"  y_train_a_int {y_train_a_int.shape}  y_dev_a_int {y_dev_a_int.shape}")
print(f"  y_train_b_int {y_train_b_int.shape}  y_dev_b_int {y_dev_b_int.shape}")

print(f"\n✓ Cell 3 complete")

In [ ]:
# ============================================================
# Cell 4: Fold-safe classical stacking
#   - TF-IDF fit INSIDE each fold (no leakage)
#   - Full train data for final model
#   - Predict on dev (official eval) + test (submission)
# ============================================================
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.ensemble import StackingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import MinMaxScaler


def build_features_fold(fit_clean, fit_meta, test_clean, test_meta):
    """FOLD-SAFE: fit TF-IDF + scaler on FIT only."""
    wv = TfidfVectorizer(analyzer="word", ngram_range=(1, 3), min_df=2,
                         max_features=30000, sublinear_tf=True)
    cv = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 6), min_df=2,
                         max_features=50000, sublinear_tf=True)
    Xw_tr = wv.fit_transform(fit_clean); Xc_tr = cv.fit_transform(fit_clean)
    Xw_te = wv.transform(test_clean);     Xc_te = cv.transform(test_clean)
    scaler = MinMaxScaler()
    m_tr = scaler.fit_transform(fit_meta); m_te = scaler.transform(test_meta)
    return (sp.hstack([Xw_tr, Xc_tr, sp.csr_matrix(m_tr)]).tocsr(),
            sp.hstack([Xw_te, Xc_te, sp.csr_matrix(m_te)]).tocsr())


def make_stack():
    """3 base models + meta-LR."""
    svc = CalibratedClassifierCV(LinearSVC(class_weight="balanced", C=0.5, max_iter=3000), cv=3)
    lr  = LogisticRegression(class_weight="balanced", C=1.0, max_iter=2000)
    cnb = ComplementNB()
    return StackingClassifier(
        estimators=[("svc", svc), ("lr", lr), ("cnb", cnb)],
        final_estimator=LogisticRegression(class_weight="balanced", max_iter=2000),
        cv=3, n_jobs=-1,
    )


def cv_eval_foldsafe(clean, meta, y, n_classes, n_splits=5):
    """Honest OOF CV: features rebuilt inside each fold."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    y = np.asarray(y)
    oof_proba = np.zeros((len(y), n_classes))
    for fold, (tr_idx, va_idx) in enumerate(skf.split(clean, y)):
        X_tr, X_va = build_features_fold(clean[tr_idx], meta[tr_idx],
                                           clean[va_idx], meta[va_idx])
        clf = make_stack(); clf.fit(X_tr, y[tr_idx])
        oof_proba[va_idx] = clf.predict_proba(X_va)
    return oof_proba


# ============================================================
# Task A — 5-fold CV + final model
# ============================================================
print("=" * 70)
print("Task A — Fold-safe 5-fold CV")
print("=" * 70)

oof_proba_a = cv_eval_foldsafe(
    train_a["Comment_clean"].values, META_TRAIN_A, y_train_a_int, len(LABELS_A)
)
oof_f1_a = f1_score(y_train_a_int, np.argmax(oof_proba_a, axis=-1), average="macro")
print(f"Task A — OOF CV Macro-F1: {oof_f1_a:.4f}")

# Final model on full train_a → predict dev_a + test_a
X_full_a, X_dev_a = build_features_fold(
    train_a["Comment_clean"].values, META_TRAIN_A,
    dev_a["Comment_clean"].values, META_DEV_A,
)
_, X_test_a = build_features_fold(
    train_a["Comment_clean"].values, META_TRAIN_A,
    test_a["Comment_clean"].values, META_TEST_A,
)
clf_a = make_stack()
clf_a.fit(X_full_a, train_a["Label"].values)

stack_proba_dev_a  = clf_a.predict_proba(X_dev_a)
stack_proba_test_a = clf_a.predict_proba(X_test_a)

f1_dev_a = f1_score(y_dev_a_int, np.argmax(stack_proba_dev_a, axis=-1), average="macro")
print(f"\nTask A — Dev F1: {f1_dev_a:.4f}")
print(classification_report(y_dev_a_int, np.argmax(stack_proba_dev_a, axis=-1),
                            target_names=LABELS_A, digits=3))


# ============================================================
# Task B — 5-fold CV + final model
# ============================================================
print("\n" + "=" * 70)
print("Task B — Fold-safe 5-fold CV")
print("=" * 70)

oof_proba_b = cv_eval_foldsafe(
    train_b["Comment_clean"].values, META_TRAIN_B, y_train_b_int, len(LABELS_B)
)
oof_f1_b = f1_score(y_train_b_int, np.argmax(oof_proba_b, axis=-1), average="macro")
print(f"Task B — OOF CV Macro-F1: {oof_f1_b:.4f}")

X_full_b, X_dev_b = build_features_fold(
    train_b["Comment_clean"].values, META_TRAIN_B,
    dev_b["Comment_clean"].values, META_DEV_B,
)
_, X_test_b = build_features_fold(
    train_b["Comment_clean"].values, META_TRAIN_B,
    test_b["Comment_clean"].values, META_TEST_B,
)
clf_b = make_stack()
clf_b.fit(X_full_b, train_b["Hate Category"].values)

stack_proba_dev_b  = clf_b.predict_proba(X_dev_b)
stack_proba_test_b = clf_b.predict_proba(X_test_b)

f1_dev_b = f1_score(y_dev_b_int, np.argmax(stack_proba_dev_b, axis=-1), average="macro")
print(f"\nTask B — Dev F1: {f1_dev_b:.4f}")
print(classification_report(y_dev_b_int, np.argmax(stack_proba_dev_b, axis=-1),
                            target_names=LABELS_B, digits=3))


# ============================================================
# Save for ensemble
# ============================================================
with open("./classical_state.pkl", "wb") as f:
    pickle.dump({
        "oof_proba_a": oof_proba_a, "oof_proba_b": oof_proba_b,
        "stack_proba_dev_a":  stack_proba_dev_a,
        "stack_proba_test_a": stack_proba_test_a,
        "stack_proba_dev_b":  stack_proba_dev_b,
        "stack_proba_test_b": stack_proba_test_b,
        "oof_f1_a": oof_f1_a, "oof_f1_b": oof_f1_b,
        "f1_dev_a": f1_dev_a, "f1_dev_b": f1_dev_b,
    }, f)

print("\n" + "=" * 70)
print("Summary")
print("=" * 70)
print(f"Task A — OOF CV F1: {oof_f1_a:.4f}  |  Dev F1: {f1_dev_a:.4f}")
print(f"Task B — OOF CV F1: {oof_f1_b:.4f}  |  Dev F1: {f1_dev_b:.4f}")
print(f"\n✓ Saved classical_state.pkl")
print(f"✓ Cell 4 complete")

# Cell 5: R-Drop trainer setup


In [ ]:
# ============================================================
# Cell 5: R-Drop trainer setup
#   Defines trainer class + factory + inference helper.
#   No training happens here — setup only (~10 sec).
# ============================================================
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from datasets import Dataset
from scipy.special import softmax

# ---- API detection (transformers 5.x compatible) ----
_TA_params = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
_Trainer_params = set(inspect.signature(Trainer.__init__).parameters.keys())
EVAL_KEY         = "eval_strategy" if "eval_strategy" in _TA_params else "evaluation_strategy"
USE_PROCESSING   = "processing_class" in _Trainer_params
USE_WARMUP_RATIO = "warmup_ratio" in _TA_params
USE_WARMUP_STEPS = "warmup_steps" in _TA_params
USE_SAVE_LIMIT   = "save_total_limit" in _TA_params

print(f"API: eval={EVAL_KEY}, processing={USE_PROCESSING}, "
      f"warmup_ratio={USE_WARMUP_RATIO}, warmup_steps={USE_WARMUP_STEPS}")


# ============================================================
# R-Drop Trainer
# ============================================================
class RDropTrainer(Trainer):
    """Trainer with R-Drop regularization + label smoothing."""
    def __init__(self, *args, class_weights=None, rdrop_alpha=0.5,
                 label_smoothing=0.1, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.rdrop_alpha = rdrop_alpha
        self.label_smoothing = label_smoothing

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        out1 = model(**inputs); out2 = model(**inputs)
        logits1 = out1.logits.view(-1, out1.logits.shape[-1])
        logits2 = out2.logits.view(-1, out2.logits.shape[-1])

        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights,
            label_smoothing=self.label_smoothing,
        )
        ce_loss = 0.5 * (loss_fct(logits1, labels.view(-1))
                       + loss_fct(logits2, labels.view(-1)))

        p1 = F.log_softmax(logits1, dim=-1)
        p2 = F.log_softmax(logits2, dim=-1)
        kl = 0.5 * (F.kl_div(p1, p2.exp(), reduction="batchmean")
                  + F.kl_div(p2, p1.exp(), reduction="batchmean"))

        loss = ce_loss + self.rdrop_alpha * kl
        return (loss, out1) if return_outputs else loss


# ============================================================
# make_rdrop_trainer factory
# ============================================================
def make_rdrop_trainer(model_name, train_df, eval_df, text_col, label_col,
                       labels, l2id, epochs, bs=8, lr=2e-5, weights=None,
                       out_dir="./out", warmup_ratio=0.1,
                       max_len=192, grad_accum=2,
                       gradient_checkpointing=True,
                       optim="adamw_torch", fp16=True,
                       rdrop_alpha=0.5, label_smoothing=0.1,
                       save_strategy="no"):
    """Memory-safe R-Drop trainer factory."""
    global tok
    if not hasattr(make_rdrop_trainer, "_cur") or make_rdrop_trainer._cur != model_name:
        tok = AutoTokenizer.from_pretrained(model_name)
        make_rdrop_trainer._cur = model_name

    def to_ds(df):
        ds = Dataset.from_pandas(
            df.reset_index(drop=True)[["id", text_col, label_col]].copy()
        )
        ds = ds.rename_column(text_col, "text")
        ds = ds.map(lambda b: tok(b["text"], truncation=True, max_length=max_len),
                    batched=True)
        ds = ds.map(lambda b: {"labels": [l2id[x] for x in b[label_col]]},
                    batched=True)
        ds = ds.remove_columns(["id", "text", label_col])
        return ds

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(labels),
        id2label={i: l for i, l in enumerate(labels)}, label2id=l2id,
    )
    if gradient_checkpointing:
        model.gradient_checkpointing_enable()
        model.config.use_cache = False

    tr_ds, va_ds = to_ds(train_df), to_ds(eval_df)

    def metrics(eval_pred):
        logits, gold = eval_pred
        pred = np.argmax(logits, axis=-1)
        return {"macro_f1": f1_score(gold, pred, average="macro"),
                "accuracy": accuracy_score(gold, pred)}

    load_best = (save_strategy != "no")
    ta_kwargs = {
        "output_dir": out_dir,
        EVAL_KEY: "epoch",
        "save_strategy": save_strategy,
        "learning_rate": lr,
        "per_device_train_batch_size": bs,
        "per_device_eval_batch_size": 16,
        "gradient_accumulation_steps": grad_accum,
        "num_train_epochs": epochs,
        "weight_decay": 0.05,
        "load_best_model_at_end": load_best,
        "metric_for_best_model": "macro_f1" if load_best else None,
        "greater_is_better": True if load_best else None,
        "fp16": fp16,
        "logging_steps": 50,
        "report_to": "none",
        "seed": SEED,
        "optim": optim,
        "gradient_checkpointing": gradient_checkpointing,
        "lr_scheduler_type": "cosine",
        "dataloader_pin_memory": False,
    }
    ta_kwargs = {k: v for k, v in ta_kwargs.items() if v is not None}

    if USE_WARMUP_RATIO:
        ta_kwargs["warmup_ratio"] = warmup_ratio
    elif USE_WARMUP_STEPS:
        eff_bs = bs * grad_accum
        steps_per_epoch = max(1, len(train_df) // eff_bs)
        total_steps = steps_per_epoch * epochs
        ta_kwargs["warmup_steps"] = max(1, int(total_steps * warmup_ratio))

    if USE_SAVE_LIMIT and save_strategy != "no":
        ta_kwargs["save_total_limit"] = 1

    args = TrainingArguments(**ta_kwargs)

    tr_kwargs = dict(
        model=model, args=args, train_dataset=tr_ds, eval_dataset=va_ds,
        data_collator=DataCollatorWithPadding(tok),
        compute_metrics=metrics, class_weights=weights,
        rdrop_alpha=rdrop_alpha, label_smoothing=label_smoothing,
    )
    tr_kwargs["processing_class" if USE_PROCESSING else "tokenizer"] = tok
    return RDropTrainer(**tr_kwargs), tr_ds, va_ds


# ---- Helper: tokenize a df for inference ----
def make_sub_dataset(df, text_col="Comment_clean", max_len=192):
    ds = Dataset.from_pandas(df[["id", text_col]].copy())
    ds = ds.rename_column(text_col, "text")
    ds = ds.map(lambda b: tok(b["text"], truncation=True, max_length=max_len),
                batched=True)
    ds = ds.remove_columns(["text"])
    return ds


# ---- GPU check ----
gc.collect(); torch.cuda.empty_cache()
free_gb, _ = [x / 1e9 for x in torch.cuda.mem_get_info()]
print(f"\nGPU free: {free_gb:.2f} GB")
print(f"✓ RDropTrainer + make_rdrop_trainer + make_sub_dataset ready")
print(f"\nNext: Cell 6 (Task A 5-fold CV with XLM-R, ~40 min)")

In [ ]:
# ============================================================
# Cell 5a: Add missing import
# ============================================================
from sklearn.metrics import accuracy_score, f1_score, classification_report
print("✓ accuracy_score imported")
print(f"✓ GPU free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

# Cell 6: Task A — XLM-R 5-fold CV with R-Drop

In [ ]:
# ============================================================
# Cell 6: Task A — XLM-R 5-fold CV with R-Drop
# ============================================================
from sklearn.model_selection import StratifiedKFold
import gc, pickle, os

# ---- Config (missing from Cell 1) ----
N_FOLDS = 5
MAX_LEN = 192

gc.collect(); torch.cuda.empty_cache()
free_gb, _ = [x / 1e9 for x in torch.cuda.mem_get_info()]
print(f"GPU free at start: {free_gb:.2f} GB")

os.makedirs("./saved_state", exist_ok=True)

XLM_R = "xlm-roberta-base"
skf_a = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Storage
oof_proba_a_xlmr = np.zeros((len(train_a), len(LABELS_A)), dtype=np.float32)
dev_probas_a     = []
test_probas_a    = []
fold_f1s_a_xlmr  = []

for fold, (tr_idx, va_idx) in enumerate(skf_a.split(train_a, y_train_a_int)):
    print(f"\n{'='*60}")
    print(f"Fold {fold+1}/{N_FOLDS}  train={len(tr_idx)}  val={len(va_idx)}")
    free_gb, _ = [x / 1e9 for x in torch.cuda.mem_get_info()]
    print(f"GPU free before fold: {free_gb:.2f} GB")
    print(f"{'='*60}")

    fold_train = train_a.iloc[tr_idx].reset_index(drop=True)
    fold_val   = train_a.iloc[va_idx].reset_index(drop=True)

    for _ in range(2): gc.collect()
    torch.cuda.empty_cache(); torch.cuda.synchronize()

    trainer_fold, _, _ = make_rdrop_trainer(
        XLM_R, fold_train, fold_val,
        text_col="Comment_clean", label_col="Label",
        labels=LABELS_A, l2id=L2ID_A,
        epochs=5,
        bs=8, grad_accum=2,
        lr=2e-5,
        weights=None,
        out_dir=f"./out_a_fold{fold}",
        max_len=MAX_LEN,
        gradient_checkpointing=True,
        optim="adamw_torch",
        fp16=True,
        rdrop_alpha=0.5, label_smoothing=0.1,
        save_strategy="no",
    )
    trainer_fold.train()

    # OOF prediction
    oof_logits = trainer_fold.predict(trainer_fold.eval_dataset).predictions
    p_oof = softmax(oof_logits, axis=-1)
    oof_proba_a_xlmr[va_idx] = p_oof

    f1_fold = f1_score(y_train_a_int[va_idx], np.argmax(p_oof, axis=-1),
                       average="macro")
    fold_f1s_a_xlmr.append(f1_fold)
    print(f"  Fold {fold+1} OOF F1: {f1_fold:.4f}")

    # Dev prediction
    dev_logits = trainer_fold.predict(
        make_sub_dataset(dev_a, "Comment_clean", MAX_LEN)
    ).predictions
    dev_probas_a.append(softmax(dev_logits, axis=-1))

    # Test prediction
    test_logits = trainer_fold.predict(
        make_sub_dataset(test_a, "Comment_clean", MAX_LEN)
    ).predictions
    test_probas_a.append(softmax(test_logits, axis=-1))

    # Teardown
    del trainer_fold, oof_logits, dev_logits, test_logits
    for _ in range(3): gc.collect()
    torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    if os.path.exists(f"./out_a_fold{fold}"):
        shutil.rmtree(f"./out_a_fold{fold}", ignore_errors=True)

    free_gb, _ = [x / 1e9 for x in torch.cuda.mem_get_info()]
    print(f"  GPU free after fold {fold+1}: {free_gb:.2f} GB")

    # Auto-save
    with open("./saved_state/task_a_xlmr_partial.pkl", "wb") as f:
        pickle.dump({
            "oof_proba_a_xlmr": oof_proba_a_xlmr,
            "dev_probas_a":     dev_probas_a,
            "test_probas_a":    test_probas_a,
            "fold_f1s_a_xlmr":  fold_f1s_a_xlmr,
        }, f)
    print(f"  ✓ Auto-saved fold {fold+1}")

# ---- Aggregate ----
DEV_PROBA_A_XLMR  = np.mean(dev_probas_a,  axis=0)
TEST_PROBA_A_XLMR = np.mean(test_probas_a, axis=0)

oof_pred = np.argmax(oof_proba_a_xlmr, axis=-1)
oof_f1_a_xlmr = f1_score(y_train_a_int, oof_pred, average="macro")
dev_f1_a_xlmr = f1_score(y_dev_a_int, np.argmax(DEV_PROBA_A_XLMR, axis=-1),
                         average="macro")

print(f"\n{'='*60}")
print(f"Task A — XLM-R 5-Fold CV Results")
print(f"{'='*60}")
print(f"Per-fold OOF F1s: {[f'{x:.4f}' for x in fold_f1s_a_xlmr]}")
print(f"Mean ± std: {np.mean(fold_f1s_a_xlmr):.4f} ± {np.std(fold_f1s_a_xlmr):.4f}")
print(f"\nOOF CV F1:  {oof_f1_a_xlmr:.4f}")
print(f"Dev F1:     {dev_f1_a_xlmr:.4f}")
print(f"{'='*60}")
print(classification_report(y_dev_a_int, np.argmax(DEV_PROBA_A_XLMR, axis=-1),
                            target_names=LABELS_A, digits=3))

# Save
with open("./saved_state/task_a_xlmr.pkl", "wb") as f:
    pickle.dump({
        "oof_proba_a_xlmr":   oof_proba_a_xlmr,
        "DEV_PROBA_A_XLMR":   DEV_PROBA_A_XLMR,
        "TEST_PROBA_A_XLMR":  TEST_PROBA_A_XLMR,
        "fold_f1s_a_xlmr":    fold_f1s_a_xlmr,
        "oof_f1_a_xlmr":      oof_f1_a_xlmr,
        "dev_f1_a_xlmr":      dev_f1_a_xlmr,
    }, f)

print(f"\n✓ Saved to ./saved_state/task_a_xlmr.pkl")
print(f"✓ Cell 6 complete")

In [ ]:
# ============================================================
# Cell 7a: Save ALL state to disk (crash-safe + restart-safe)
#   Saves: dataframes, labels, meta features, probas, configs
# ============================================================
import pickle, os

SAVE_DIR = "/kaggle/working/saved_session"
os.makedirs(SAVE_DIR, exist_ok=True)

# Everything needed to resume
session_state = {
    # ---- Data ----
    "train_a": train_a, "dev_a": dev_a, "test_a": test_a,
    "train_b": train_b, "dev_b": dev_b, "test_b": test_b,

    # ---- Meta features ----
    "META_TRAIN_A": META_TRAIN_A, "META_DEV_A": META_DEV_A, "META_TEST_A": META_TEST_A,
    "META_TRAIN_B": META_TRAIN_B, "META_DEV_B": META_DEV_B, "META_TEST_B": META_TEST_B,

    # ---- Labels ----
    "LABELS_A": LABELS_A, "L2ID_A": L2ID_A, "ID2L_A": ID2L_A,
    "LABELS_B": LABELS_B, "L2ID_B": L2ID_B, "ID2L_B": ID2L_B,
    "y_train_a_int": y_train_a_int, "y_dev_a_int": y_dev_a_int,
    "y_train_b_int": y_train_b_int, "y_dev_b_int": y_dev_b_int,

    # ---- Class weights ----
    "w_b": w_b,

    # ---- Config ----
    "SEED": SEED, "MAX_LEN": MAX_LEN, "N_FOLDS": N_FOLDS,
}

with open(f"{SAVE_DIR}/session_state.pkl", "wb") as f:
    pickle.dump(session_state, f)
print(f"✓ session_state.pkl saved ({os.path.getsize(f'{SAVE_DIR}/session_state.pkl')/1e6:.2f} MB)")

# Classical state already saved in Cell 4 → copy it to safe dir
import shutil
if os.path.exists("./classical_state.pkl"):
    shutil.copy("./classical_state.pkl", f"{SAVE_DIR}/classical_state.pkl")
    print(f"✓ classical_state.pkl copied")

# XLM-R Task A saved in Cell 6 → copy
if os.path.exists("./saved_state/task_a_xlmr.pkl"):
    shutil.copy("./saved_state/task_a_xlmr.pkl", f"{SAVE_DIR}/task_a_xlmr.pkl")
    print(f"✓ task_a_xlmr.pkl copied")

# ---- Verify ----
print(f"\n{save_dir if False else SAVE_DIR} contents:")
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(f"{SAVE_DIR}/{f}")
    print(f"  📄 {f}  ({size/1024:.1f} KB)")

print(f"\n✓ Cell 7a complete — state safe on disk")

In [4]:
# ============================================================
# RESTORE Cell 1: Environment (fast, ~1 min)
# ============================================================
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q -U transformers datasets accelerate scikit-learn emoji

import random, inspect, shutil, zipfile, itertools, re, html, gc, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
from scipy.special import softmax
import emoji
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.ensemble import StackingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import MinMaxScaler
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from datasets import Dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
free_gb, _ = [x / 1e9 for x in torch.cuda.mem_get_info()]
print(f"GPU free: {free_gb:.2f} GB")
print("✓ Restore Cell 1 done")

Device: cuda
GPU free: 15.53 GB
✓ Restore Cell 1 done


In [6]:
# ============================================================
# Cell A1: Deep inspect isratjahanmoucat dataset
# ============================================================
import os

base = "/kaggle/input/datasets/isratjahanmoucat"
if os.path.exists(base):
    print(f"Contents of {base}:")
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if not d.startswith(".")]
        level = root.replace(base, "").count(os.sep)
        indent = "  " * level
        print(f"{indent}📁 {os.path.basename(root)}/")
        for f in files:
            fp = os.path.join(root, f)
            sz = os.path.getsize(fp)
            print(f"{indent}  📄 {f}  ({sz/1024:.1f} KB)")
else:
    print(f"✗ {base} not found")

# Also check the sibling folder
base2 = "/kaggle/input/isratjahanmoucat"
if os.path.exists(base2):
    print(f"\nContents of {base2}:")
    for root, dirs, files in os.walk(base2):
        dirs[:] = [d for d in dirs if not d.startswith(".")]
        level = root.replace(base2, "").count(os.sep)
        indent = "  " * level
        print(f"{indent}📁 {os.path.basename(root)}/")
        for f in files:
            fp = os.path.join(root, f)
            sz = os.path.getsize(fp)
            print(f"{indent}  📄 {f}  ({sz/1024:.1f} KB)")

Contents of /kaggle/input/datasets/isratjahanmoucat:
📁 isratjahanmoucat/
  📁 hastika-output/
    📄 classical_state.pkl  (311.6 KB)
    📁 Hastika-ICON2026/
      📄 LICENSE_NOTE.md  (0.6 KB)
      📄 README.md  (3.9 KB)
      📄 FORMAT.md  (0.5 KB)
      📁 starting_kit/
        📄 sample_task_b_predictions.csv  (0.1 KB)
        📄 README.md  (0.4 KB)
        📄 sample_task_a_predictions.csv  (0.1 KB)
      📁 data/
        📄 hastika_multiclass_test.csv  (36.0 KB)
        📄 hastika_binary_test.csv  (68.3 KB)
        📄 multiclass_train.csv  (320.4 KB)
        📄 hastika_multiclass_validation.csv  (41.4 KB)
        📄 hastika_binary_validation.csv  (75.4 KB)
        📄 binary_train.csv  (584.2 KB)
      📁 baseline/
        📄 baseline.py  (1.4 KB)
    📁 saved_state/
      📄 task_a_xlmr.pkl  (63.3 KB)
      📄 task_a_xlmr_partial.pkl  (114.0 KB)
    📁 saved_session/
      📄 task_a_xlmr.pkl  (63.3 KB)
      📄 session_state.pkl  (2998.6 KB)
      📄 classical_state.pkl  (311.6 KB)


In [8]:
# ============================================================
# RESTORE-1 (FINAL): Load all state from correct path
# ============================================================
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q -U transformers datasets accelerate scikit-learn emoji

import random, inspect, shutil, zipfile, itertools, re, html, gc, pickle
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
import scipy.sparse as sp
from scipy.special import softmax
import emoji
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.ensemble import StackingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import MinMaxScaler
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from datasets import Dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_FOLDS = 5
MAX_LEN = 192
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

# ============================================================
# Path (from user)
# ============================================================
DATASET_ROOT = "/kaggle/input/datasets/isratjahanmoucat/hastika-output"
print(f"\nDataset root: {DATASET_ROOT}")
print(f"Exists: {os.path.exists(DATASET_ROOT)}")

# ---- List contents ----
print(f"\nDataset contents:")
for f in sorted(os.listdir(DATASET_ROOT)):
    fp = f"{DATASET_ROOT}/{f}"
    if os.path.isdir(fp):
        print(f"  📁 {f}/")
        for sub in sorted(os.listdir(fp)):
            sp2 = f"{fp}/{sub}"
            sz = os.path.getsize(sp2) if os.path.isfile(sp2) else 0
            print(f"      {sub}  ({sz/1024:.1f} KB)")
    else:
        sz = os.path.getsize(fp)
        print(f"  📄 {f}  ({sz/1024:.1f} KB)")

# ============================================================
# Load session_state.pkl
# ============================================================
sess_path = f"{DATASET_ROOT}/saved_session/session_state.pkl"
print(f"\nLoading: {sess_path}")
with open(sess_path, "rb") as f:
    sess = pickle.load(f)
for k, v in sess.items():
    globals()[k] = v

print(f"✓ Loaded {len(sess)} items")
for k in sorted(sess.keys()):
    v = sess[k]
    if isinstance(v, np.ndarray):
        print(f"    {k:25s} shape={v.shape}")
    elif isinstance(v, pd.DataFrame):
        print(f"    {k:25s} shape={v.shape}")
    else:
        print(f"    {k:25s} {type(v).__name__}")

# ============================================================
# Load classical_state.pkl
# ============================================================
cl_path = f"{DATASET_ROOT}/saved_session/classical_state.pkl"
print(f"\nLoading: {cl_path}")
with open(cl_path, "rb") as f:
    cl = pickle.load(f)
for k, v in cl.items():
    globals()[k] = v

print(f"✓ Loaded {len(cl)} items")
for k in sorted(cl.keys()):
    v = cl[k]
    if isinstance(v, np.ndarray):
        print(f"    {k:25s} shape={v.shape}")
    else:
        print(f"    {k:25s} {type(v).__name__}")

# ============================================================
# Load task_a_xlmr.pkl
# ============================================================
xa_path = f"{DATASET_ROOT}/saved_session/task_a_xlmr.pkl"
print(f"\nLoading: {xa_path}")
with open(xa_path, "rb") as f:
    xa = pickle.load(f)
for k, v in xa.items():
    globals()[k] = v

print(f"✓ Loaded {len(xa)} items")
for k in sorted(xa.keys()):
    v = xa[k]
    if isinstance(v, np.ndarray):
        print(f"    {k:25s} shape={v.shape}")
    else:
        print(f"    {k:25s} {type(v).__name__}")

print(f"\n✓ RESTORE-1 complete")

Device: cuda
GPU: Tesla T4
GPU free: 15.53 GB

Dataset root: /kaggle/input/datasets/isratjahanmoucat/hastika-output
Exists: True

Dataset contents:
  📁 .virtual_documents/
      __notebook_source__.ipynb  (30.4 KB)
  📁 Hastika-ICON2026/
      .git  (0.0 KB)
      FORMAT.md  (0.5 KB)
      LICENSE_NOTE.md  (0.6 KB)
      README.md  (3.9 KB)
      baseline  (0.0 KB)
      data  (0.0 KB)
      starting_kit  (0.0 KB)
  📄 classical_state.pkl  (311.6 KB)
  📁 saved_session/
      classical_state.pkl  (311.6 KB)
      session_state.pkl  (2998.6 KB)
      task_a_xlmr.pkl  (63.3 KB)
  📁 saved_state/
      task_a_xlmr.pkl  (63.3 KB)
      task_a_xlmr_partial.pkl  (114.0 KB)

Loading: /kaggle/input/datasets/isratjahanmoucat/hastika-output/saved_session/session_state.pkl
✓ Loaded 26 items
    ID2L_A                    dict
    ID2L_B                    dict
    L2ID_A                    dict
    L2ID_B                    dict
    LABELS_A                  list
    LABELS_B                  list
   

In [9]:
# ============================================================
# RESUME-2: Helper functions
# ============================================================

# ---- W_B as torch tensor (from w_b numpy array) ----
W_B = torch.tensor(w_b, dtype=torch.float).to(DEVICE)
print(f"W_B: {W_B.cpu().numpy().round(3)}")

# ---- API detection ----
_TA_params = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
_Trainer_params = set(inspect.signature(Trainer.__init__).parameters.keys())
EVAL_KEY         = "eval_strategy" if "eval_strategy" in _TA_params else "evaluation_strategy"
USE_PROCESSING   = "processing_class" in _Trainer_params
USE_WARMUP_RATIO = "warmup_ratio" in _TA_params
USE_WARMUP_STEPS = "warmup_steps" in _TA_params
USE_SAVE_LIMIT   = "save_total_limit" in _TA_params
print(f"API: eval={EVAL_KEY}, processing={USE_PROCESSING}")


class RDropTrainer(Trainer):
    """Trainer with R-Drop regularization + label smoothing."""
    def __init__(self, *args, class_weights=None, rdrop_alpha=0.5,
                 label_smoothing=0.1, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.rdrop_alpha = rdrop_alpha
        self.label_smoothing = label_smoothing

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        out1 = model(**inputs); out2 = model(**inputs)
        logits1 = out1.logits.view(-1, out1.logits.shape[-1])
        logits2 = out2.logits.view(-1, out2.logits.shape[-1])

        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights,
                                        label_smoothing=self.label_smoothing)
        ce_loss = 0.5 * (loss_fct(logits1, labels.view(-1))
                       + loss_fct(logits2, labels.view(-1)))

        p1 = F.log_softmax(logits1, dim=-1)
        p2 = F.log_softmax(logits2, dim=-1)
        kl = 0.5 * (F.kl_div(p1, p2.exp(), reduction="batchmean")
                  + F.kl_div(p2, p1.exp(), reduction="batchmean"))

        loss = ce_loss + self.rdrop_alpha * kl
        return (loss, out1) if return_outputs else loss


def make_rdrop_trainer(model_name, train_df, eval_df, text_col, label_col,
                       labels, l2id, epochs, bs=8, lr=2e-5, weights=None,
                       out_dir="./out", warmup_ratio=0.1, max_len=192,
                       grad_accum=2, gradient_checkpointing=True,
                       optim="adamw_torch", fp16=True,
                       rdrop_alpha=0.5, label_smoothing=0.1,
                       save_strategy="no"):
    global tok
    if not hasattr(make_rdrop_trainer, "_cur") or make_rdrop_trainer._cur != model_name:
        tok = AutoTokenizer.from_pretrained(model_name)
        make_rdrop_trainer._cur = model_name

    def to_ds(df):
        ds = Dataset.from_pandas(
            df.reset_index(drop=True)[["id", text_col, label_col]].copy()
        )
        ds = ds.rename_column(text_col, "text")
        ds = ds.map(lambda b: tok(b["text"], truncation=True, max_length=max_len),
                    batched=True)
        ds = ds.map(lambda b: {"labels": [l2id[x] for x in b[label_col]]},
                    batched=True)
        ds = ds.remove_columns(["id", "text", label_col])
        return ds

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(labels),
        id2label={i: l for i, l in enumerate(labels)}, label2id=l2id,
    )
    if gradient_checkpointing:
        model.gradient_checkpointing_enable()
        model.config.use_cache = False

    tr_ds, va_ds = to_ds(train_df), to_ds(eval_df)

    def metrics(eval_pred):
        logits, gold = eval_pred
        pred = np.argmax(logits, axis=-1)
        return {"macro_f1": f1_score(gold, pred, average="macro"),
                "accuracy": accuracy_score(gold, pred)}

    load_best = (save_strategy != "no")
    ta_kwargs = {
        "output_dir": out_dir, EVAL_KEY: "epoch", "save_strategy": save_strategy,
        "learning_rate": lr, "per_device_train_batch_size": bs,
        "per_device_eval_batch_size": 16,
        "gradient_accumulation_steps": grad_accum,
        "num_train_epochs": epochs, "weight_decay": 0.05,
        "load_best_model_at_end": load_best,
        "metric_for_best_model": "macro_f1" if load_best else None,
        "greater_is_better": True if load_best else None,
        "fp16": fp16, "logging_steps": 50, "report_to": "none",
        "seed": SEED, "optim": optim,
        "gradient_checkpointing": gradient_checkpointing,
        "lr_scheduler_type": "cosine", "dataloader_pin_memory": False,
    }
    ta_kwargs = {k: v for k, v in ta_kwargs.items() if v is not None}
    if USE_WARMUP_RATIO: ta_kwargs["warmup_ratio"] = warmup_ratio
    elif USE_WARMUP_STEPS:
        eff_bs = bs * grad_accum
        spe = max(1, len(train_df) // eff_bs)
        ts = spe * epochs
        ta_kwargs["warmup_steps"] = max(1, int(ts * warmup_ratio))
    if USE_SAVE_LIMIT and save_strategy != "no":
        ta_kwargs["save_total_limit"] = 1

    args = TrainingArguments(**ta_kwargs)
    tr_kwargs = dict(model=model, args=args, train_dataset=tr_ds, eval_dataset=va_ds,
                     data_collator=DataCollatorWithPadding(tok),
                     compute_metrics=metrics, class_weights=weights,
                     rdrop_alpha=rdrop_alpha, label_smoothing=label_smoothing)
    tr_kwargs["processing_class" if USE_PROCESSING else "tokenizer"] = tok
    return RDropTrainer(**tr_kwargs), tr_ds, va_ds


def make_sub_dataset(df, text_col="Comment_clean", max_len=192):
    ds = Dataset.from_pandas(df[["id", text_col]].copy())
    ds = ds.rename_column(text_col, "text")
    ds = ds.map(lambda b: tok(b["text"], truncation=True, max_length=max_len),
                batched=True)
    ds = ds.remove_columns(["text"])
    return ds


gc.collect(); torch.cuda.empty_cache()
print(f"\n✓ RESUME-2 complete")
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

W_B: [0.387 2.831 1.173 0.942 1.378 2.382]
API: eval=eval_strategy, processing=True

✓ RESUME-2 complete
GPU free: 15.52 GB


In [10]:
# ============================================================
# Cell 7: Task B — XLM-R 5-fold CV with R-Drop
#   Uses class weights (W_B) for imbalance.
#   Saves to /kaggle/working/saved_session/ (crash-safe)
# ============================================================
from sklearn.model_selection import StratifiedKFold
import gc, pickle, os

gc.collect(); torch.cuda.empty_cache()
free_gb, _ = [x / 1e9 for x in torch.cuda.mem_get_info()]
print(f"GPU free at start: {free_gb:.2f} GB")

os.makedirs("/kaggle/working/saved_session", exist_ok=True)

XLM_R = "xlm-roberta-base"
skf_b = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Storage
oof_proba_b_xlmr = np.zeros((len(train_b), len(LABELS_B)), dtype=np.float32)
dev_probas_b     = []   # list of (395, 6)
test_probas_b    = []   # list of (396, 6)
fold_f1s_b_xlmr  = []

for fold, (tr_idx, va_idx) in enumerate(skf_b.split(train_b, y_train_b_int)):
    print(f"\n{'='*60}")
    print(f"Fold {fold+1}/{N_FOLDS}  train={len(tr_idx)}  val={len(va_idx)}")
    free_gb, _ = [x / 1e9 for x in torch.cuda.mem_get_info()]
    print(f"GPU free before fold: {free_gb:.2f} GB")
    print(f"{'='*60}")

    fold_train = train_b.iloc[tr_idx].reset_index(drop=True)
    fold_val   = train_b.iloc[va_idx].reset_index(drop=True)

    for _ in range(2): gc.collect()
    torch.cuda.empty_cache(); torch.cuda.synchronize()

    trainer_fold, _, _ = make_rdrop_trainer(
        XLM_R, fold_train, fold_val,
        text_col="Comment_clean", label_col="Hate Category",
        labels=LABELS_B, l2id=L2ID_B,
        epochs=5,
        bs=8, grad_accum=2,
        lr=2e-5,
        weights=W_B,
        out_dir=f"./out_b_fold{fold}",
        max_len=MAX_LEN,
        gradient_checkpointing=True,
        optim="adamw_torch",
        fp16=True,
        rdrop_alpha=0.5, label_smoothing=0.1,
        save_strategy="no",
    )
    trainer_fold.train()

    # OOF
    oof_logits = trainer_fold.predict(trainer_fold.eval_dataset).predictions
    p_oof = softmax(oof_logits, axis=-1)
    oof_proba_b_xlmr[va_idx] = p_oof

    f1_fold = f1_score(y_train_b_int[va_idx], np.argmax(p_oof, axis=-1),
                       average="macro")
    fold_f1s_b_xlmr.append(f1_fold)
    print(f"  Fold {fold+1} OOF F1: {f1_fold:.4f}")

    # Dev
    dev_logits = trainer_fold.predict(
        make_sub_dataset(dev_b, "Comment_clean", MAX_LEN)
    ).predictions
    dev_probas_b.append(softmax(dev_logits, axis=-1))

    # Test
    test_logits = trainer_fold.predict(
        make_sub_dataset(test_b, "Comment_clean", MAX_LEN)
    ).predictions
    test_probas_b.append(softmax(test_logits, axis=-1))

    # Teardown
    del trainer_fold, oof_logits, dev_logits, test_logits
    for _ in range(3): gc.collect()
    torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    if os.path.exists(f"./out_b_fold{fold}"):
        shutil.rmtree(f"./out_b_fold{fold}", ignore_errors=True)

    free_gb, _ = [x / 1e9 for x in torch.cuda.mem_get_info()]
    print(f"  GPU free after fold {fold+1}: {free_gb:.2f} GB")

    # Auto-save
    with open("/kaggle/working/saved_session/task_b_xlmr_partial.pkl", "wb") as f:
        pickle.dump({
            "oof_proba_b_xlmr": oof_proba_b_xlmr,
            "dev_probas_b":     dev_probas_b,
            "test_probas_b":    test_probas_b,
            "fold_f1s_b_xlmr":  fold_f1s_b_xlmr,
        }, f)
    print(f"  ✓ Auto-saved fold {fold+1}")

# ---- Aggregate ----
DEV_PROBA_B_XLMR  = np.mean(dev_probas_b,  axis=0)
TEST_PROBA_B_XLMR = np.mean(test_probas_b, axis=0)

oof_pred = np.argmax(oof_proba_b_xlmr, axis=-1)
oof_f1_b_xlmr = f1_score(y_train_b_int, oof_pred, average="macro")
dev_f1_b_xlmr = f1_score(y_dev_b_int, np.argmax(DEV_PROBA_B_XLMR, axis=-1),
                         average="macro")

print(f"\n{'='*60}")
print(f"Task B — XLM-R 5-Fold CV Results")
print(f"{'='*60}")
print(f"Per-fold OOF F1s: {[f'{x:.4f}' for x in fold_f1s_b_xlmr]}")
print(f"Mean ± std: {np.mean(fold_f1s_b_xlmr):.4f} ± {np.std(fold_f1s_b_xlmr):.4f}")
print(f"\nOOF CV F1:  {oof_f1_b_xlmr:.4f}")
print(f"Dev F1:     {dev_f1_b_xlmr:.4f}")
print(f"{'='*60}")
print(classification_report(y_dev_b_int, np.argmax(DEV_PROBA_B_XLMR, axis=-1),
                            target_names=LABELS_B, digits=3))

# ---- Save final ----
with open("/kaggle/working/saved_session/task_b_xlmr.pkl", "wb") as f:
    pickle.dump({
        "oof_proba_b_xlmr":   oof_proba_b_xlmr,
        "DEV_PROBA_B_XLMR":   DEV_PROBA_B_XLMR,
        "TEST_PROBA_B_XLMR":  TEST_PROBA_B_XLMR,
        "fold_f1s_b_xlmr":    fold_f1s_b_xlmr,
        "oof_f1_b_xlmr":      oof_f1_b_xlmr,
        "dev_f1_b_xlmr":      dev_f1_b_xlmr,
    }, f)

print(f"\n✓ Saved to /kaggle/working/saved_session/task_b_xlmr.pkl")
print(f"✓ Cell 7 complete")

GPU free at start: 15.52 GB

Fold 1/5  train=2527  val=632
GPU free before fold: 15.52 GB


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

... (52 outputs left out)

  Fold 5 OOF F1: 0.5389


Map:   0%|          | 0/395 [00:00<?, ? examples/s]

Map:   0%|          | 0/396 [00:00<?, ? examples/s]

  GPU free after fold 5: 15.34 GB
  ✓ Auto-saved fold 5

Task B — XLM-R 5-Fold CV Results
Per-fold OOF F1s: ['0.5107', '0.5020', '0.5120', '0.5100', '0.5389']
Mean ± std: 0.5147 ± 0.0126

OOF CV F1:  0.5151
Dev F1:     0.5321
               precision    recall  f1-score   support

       Gender      0.780     0.647     0.707       170
Geo-political      0.441     0.652     0.526        23
       Others      0.413     0.339     0.373        56
    Political      0.675     0.771     0.720        70
     Religion      0.684     0.542     0.605        48
     Violence      0.196     0.393     0.262        28

     accuracy                          0.595       395
    macro avg      0.532     0.557     0.532       395
 weighted avg      0.637     0.595     0.608       395


✓ Saved to /kaggle/working/saved_session/task_b_xlmr.pkl
✓ Cell 7 complete


In [11]:
# ============================================================
# Cell 8: Final Ensemble + Generate test predictions
#   For both Task A and Task B:
#     - Weight search on dev
#     - Optional threshold tuning (Task A)
#     - Generate test predictions
#     - Save submission files
# ============================================================
import pickle, itertools, os, zipfile
from scipy.optimize import minimize
import numpy as np, pandas as pd

# ============================================================
# TASK A — Ensemble
# ============================================================
print("=" * 70)
print("TASK A — Final Ensemble")
print("=" * 70)

y_dev_a_int = np.array([L2ID_A[x] for x in dev_a["Label"].values])

# Members
members_dev_a = {
    "classical": stack_proba_dev_a,     # (806, 2)
    "xlmr":      DEV_PROBA_A_XLMR,      # (806, 2)
}
members_test_a = {
    "classical": stack_proba_test_a,    # (806, 2)
    "xlmr":      TEST_PROBA_A_XLMR,     # (806, 2)
}

print("Individual members on dev_a:")
for name, p in members_dev_a.items():
    f1 = f1_score(y_dev_a_int, np.argmax(p, axis=-1), average="macro")
    print(f"  {name:12s} F1={f1:.4f}")

# Grid search weights (2 models)
best_f1_a, best_w_a = 0, None
for w_c in np.arange(0.0, 1.01, 0.05):
    w_x = 1.0 - w_c
    ens = w_c * members_dev_a["classical"] + w_x * members_dev_a["xlmr"]
    f1 = f1_score(y_dev_a_int, np.argmax(ens, axis=-1), average="macro")
    if f1 > best_f1_a:
        best_f1_a, best_w_a = f1, (w_c, w_x)

w_c_a, w_x_a = best_w_a
print(f"\nBest weights: classical={w_c_a:.2f}, xlmr={w_x_a:.2f}")
print(f"Task A ensemble dev F1: {best_f1_a:.4f}")

# Threshold tuning on dev
ens_dev_a = w_c_a * members_dev_a["classical"] + w_x_a * members_dev_a["xlmr"]
best_thresh_a, best_f1_a_th = 0.5, best_f1_a
for t in np.arange(0.35, 0.66, 0.005):
    preds_t = (ens_dev_a[:, L2ID_A["Non-Hate"]] >= t).astype(int)
    f1 = f1_score(y_dev_a_int, preds_t, average="macro")
    if f1 > best_f1_a_th:
        best_f1_a_th, best_thresh_a = f1, t

print(f"Best threshold: {best_thresh_a:.3f}")
print(f"Task A ensemble + threshold: {best_f1_a_th:.4f}")

# Final test predictions for Task A
ens_test_a = w_c_a * members_test_a["classical"] + w_x_a * members_test_a["xlmr"]
preds_a_test = (ens_test_a[:, L2ID_A["Non-Hate"]] >= best_thresh_a).astype(int)
submission_a = pd.DataFrame({
    "id":    test_a["id"].values,
    "label": [ID2L_A[i] for i in preds_a_test],
})

print(f"\nTask A submission shape: {submission_a.shape}")
print(f"Distribution:\n{submission_a['label'].value_counts().to_string()}")


# ============================================================
# TASK B — Ensemble
# ============================================================
print("\n" + "=" * 70)
print("TASK B — Final Ensemble")
print("=" * 70)

y_dev_b_int = np.array([L2ID_B[x] for x in dev_b["Hate Category"].values])

members_dev_b = {
    "classical": stack_proba_dev_b,     # (395, 6)
    "xlmr":      DEV_PROBA_B_XLMR,      # (395, 6)
}
members_test_b = {
    "classical": stack_proba_test_b,    # (396, 6)
    "xlmr":      TEST_PROBA_B_XLMR,     # (396, 6)
}

print("Individual members on dev_b:")
for name, p in members_dev_b.items():
    f1 = f1_score(y_dev_b_int, np.argmax(p, axis=-1), average="macro")
    print(f"  {name:12s} F1={f1:.4f}")

best_f1_b, best_w_b = 0, None
for w_c in np.arange(0.0, 1.01, 0.05):
    w_x = 1.0 - w_c
    ens = w_c * members_dev_b["classical"] + w_x * members_dev_b["xlmr"]
    f1 = f1_score(y_dev_b_int, np.argmax(ens, axis=-1), average="macro")
    if f1 > best_f1_b:
        best_f1_b, best_w_b = f1, (w_c, w_x)

w_c_b, w_x_b = best_w_b
print(f"\nBest weights: classical={w_c_b:.2f}, xlmr={w_x_b:.2f}")
print(f"Task B ensemble dev F1: {best_f1_b:.4f}")

# Final test predictions
ens_test_b = w_c_b * members_test_b["classical"] + w_x_b * members_test_b["xlmr"]
preds_b_test = np.argmax(ens_test_b, axis=-1)
submission_b = pd.DataFrame({
    "id":    test_b["id"].values,
    "label": [ID2L_B[i] for i in preds_b_test],
})

print(f"\nTask B submission shape: {submission_b.shape}")
print(f"Distribution:\n{submission_b['label'].value_counts().to_string()}")


# ============================================================
# SAVE SUBMISSION FILES
# ============================================================
print("\n" + "=" * 70)
print("SAVING SUBMISSIONS")
print("=" * 70)

out_dir = "/kaggle/working/submissions"
os.makedirs(out_dir, exist_ok=True)

# CSV files
submission_a.to_csv(f"{out_dir}/predictions_task_a.csv", index=False)
submission_b.to_csv(f"{out_dir}/predictions_task_b.csv", index=False)

# Also save as "sample" format names (in case Google Form expects specific names)
submission_a.to_csv(f"{out_dir}/sample_task_a_predictions.csv", index=False)
submission_b.to_csv(f"{out_dir}/sample_task_b_predictions.csv", index=False)

print(f"✓ Saved CSVs to {out_dir}/")
for f in sorted(os.listdir(out_dir)):
    sz = os.path.getsize(f"{out_dir}/{f}")
    print(f"    {f}  ({sz/1024:.1f} KB)")

# Save state
with open("/kaggle/working/saved_session/final_ensemble.pkl", "wb") as f:
    pickle.dump({
        "best_f1_a": best_f1_a_th, "best_w_a": best_w_a, "best_thresh_a": best_thresh_a,
        "best_f1_b": best_f1_b,    "best_w_b": best_w_b,
        "submission_a": submission_a, "submission_b": submission_b,
    }, f)

print(f"\n✓ Saved final_ensemble.pkl")

# ============================================================
# SUMMARY
# ============================================================
print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
print(f"Task A — Dev F1 (ensemble+threshold): {best_f1_a_th:.4f}")
print(f"  Weights: classical={w_c_a:.2f}, xlmr={w_x_a:.2f}  Threshold={best_thresh_a:.3f}")
print(f"\nTask B — Dev F1 (ensemble): {best_f1_b:.4f}")
print(f"  Weights: classical={w_c_b:.2f}, xlmr={w_x_b:.2f}")

print(f"\n✓ Cell 8 complete — submissions ready")

TASK A — Final Ensemble
Individual members on dev_a:
  classical    F1=0.8002
  xlmr         F1=0.8002

Best weights: classical=0.50, xlmr=0.50
Task A ensemble dev F1: 0.8213
Best threshold: 0.500
Task A ensemble + threshold: 0.8213

Task A submission shape: (806, 2)
Distribution:
label
Hate        421
Non-Hate    385

TASK B — Final Ensemble
Individual members on dev_b:
  classical    F1=0.6648
  xlmr         F1=0.5321

Best weights: classical=1.00, xlmr=0.00
Task B ensemble dev F1: 0.6648

Task B submission shape: (396, 2)
Distribution:
label
Gender           156
Political         67
Others            62
Religion          42
Violence          37
Geo-political     32

SAVING SUBMISSIONS
✓ Saved CSVs to /kaggle/working/submissions/
    predictions_task_a.csv  (9.3 KB)
    predictions_task_b.csv  (5.2 KB)
    sample_task_a_predictions.csv  (9.3 KB)
    sample_task_b_predictions.csv  (5.2 KB)

✓ Saved final_ensemble.pkl

FINAL SUMMARY
Task A — Dev F1 (ensemble+threshold): 0.8213
  Weight

In [13]:
# ============================================================
# Cell 9 (FIXED): Verify submission format
# ============================================================
import pandas as pd

# Dataset path (from earlier)
DS_ROOT = "/kaggle/input/datasets/isratjahanmoucat/hastika-output/Hastika-ICON2026"

# ---- Sample files ----
sample_a = pd.read_csv(f"{DS_ROOT}/starting_kit/sample_task_a_predictions.csv")
sample_b = pd.read_csv(f"{DS_ROOT}/starting_kit/sample_task_b_predictions.csv")

print("=" * 70)
print("Sample Task A format")
print("=" * 70)
print(f"  Columns: {list(sample_a.columns)}")
print(f"  Shape: {sample_a.shape}")
print(sample_a.head(3))
print(f"  Labels: {sample_a['label'].unique().tolist()}")

print("\n" + "=" * 70)
print("Sample Task B format")
print("=" * 70)
print(f"  Columns: {list(sample_b.columns)}")
print(f"  Shape: {sample_b.shape}")
print(sample_b.head(3))
print(f"  Labels: {sample_b['label'].unique().tolist()}")

# ---- Our submissions ----
our_a = pd.read_csv("/kaggle/working/submissions/predictions_task_a.csv")
our_b = pd.read_csv("/kaggle/working/submissions/predictions_task_b.csv")

print("\n" + "=" * 70)
print("Our Task A submission")
print("=" * 70)
print(f"  Columns: {list(our_a.columns)}")
print(f"  Shape: {our_a.shape}")
print(our_a.head(3))
print(f"  Labels: {our_a['label'].unique().tolist()}")

print("\n" + "=" * 70)
print("Our Task B submission")
print("=" * 70)
print(f"  Columns: {list(our_b.columns)}")
print(f"  Shape: {our_b.shape}")
print(our_b.head(3))
print(f"  Labels: {our_b['label'].unique().tolist()}")

# ---- Verify IDs ----
test_a_ids = set(test_a["id"].values)
test_b_ids = set(test_b["id"].values)
print(f"\nID check:")
print(f"  Task A: test={len(test_a_ids)}, submission={len(set(our_a['id']))}, match={test_a_ids == set(our_a['id'])}")
print(f"  Task B: test={len(test_b_ids)}, submission={len(set(our_b['id']))}, match={test_b_ids == set(our_b['id'])}")

Sample Task A format
  Columns: ['id', 'label']
  Shape: (5, 2)
     id     label
0  4186  Non-Hate
1  5693      Hate
2  6825  Non-Hate
  Labels: ['Non-Hate', 'Hate']

Sample Task B format
  Columns: ['id', 'label']
  Shape: (5, 2)
     id      label
0  4204     Gender
1  6264  Political
2  5274   Religion
  Labels: ['Gender', 'Political', 'Religion', 'Geo-political', 'Violence']

Our Task A submission
  Columns: ['id', 'label']
  Shape: (806, 2)
     id     label
0  5820      Hate
1  5800      Hate
2  2707  Non-Hate
  Labels: ['Hate', 'Non-Hate']

Our Task B submission
  Columns: ['id', 'label']
  Shape: (396, 2)
     id     label
0  6820    Others
1  6507    Others
2  1697  Religion
  Labels: ['Others', 'Religion', 'Gender', 'Violence', 'Political', 'Geo-political']

ID check:
  Task A: test=806, submission=806, match=True
  Task B: test=396, submission=396, match=True


# Cell 11: Task A — MuRIL 5-fold CV -Same pattern as XLM-R (Cell 6). MuRIL is Kannada-specific.


In [14]:
# ============================================================
# Cell 11: Task A — MuRIL 5-fold CV
#   Same pattern as XLM-R (Cell 6). MuRIL is Kannada-specific.
#   Saves to /kaggle/working/saved_session/
# ============================================================
from sklearn.model_selection import StratifiedKFold
import gc, pickle, os

gc.collect(); torch.cuda.empty_cache()
free_gb, _ = [x / 1e9 for x in torch.cuda.mem_get_info()]
print(f"GPU free at start: {free_gb:.2f} GB")

os.makedirs("/kaggle/working/saved_session", exist_ok=True)

MURIL = "google/muril-base-cased"
skf_a = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_proba_a_muril = np.zeros((len(train_a), len(LABELS_A)), dtype=np.float32)
dev_probas_a_muril  = []
test_probas_a_muril = []
fold_f1s_a_muril    = []

for fold, (tr_idx, va_idx) in enumerate(skf_a.split(train_a, y_train_a_int)):
    print(f"\n{'='*60}")
    print(f"Fold {fold+1}/{N_FOLDS}  train={len(tr_idx)}  val={len(va_idx)}")
    free_gb, _ = [x / 1e9 for x in torch.cuda.mem_get_info()]
    print(f"GPU free before fold: {free_gb:.2f} GB")
    print(f"{'='*60}")

    fold_train = train_a.iloc[tr_idx].reset_index(drop=True)
    fold_val   = train_a.iloc[va_idx].reset_index(drop=True)

    for _ in range(2): gc.collect()
    torch.cuda.empty_cache(); torch.cuda.synchronize()

    trainer_fold, _, _ = make_rdrop_trainer(
        MURIL, fold_train, fold_val,
        text_col="Comment_clean", label_col="Label",
        labels=LABELS_A, l2id=L2ID_A,
        epochs=5,
        bs=8, grad_accum=2,
        lr=2e-5,
        weights=None,
        out_dir=f"./out_a_muril_fold{fold}",
        max_len=MAX_LEN,
        gradient_checkpointing=True,
        optim="adamw_torch",
        fp16=True,
        rdrop_alpha=0.5, label_smoothing=0.1,
        save_strategy="no",
    )
    trainer_fold.train()

    # OOF
    oof_logits = trainer_fold.predict(trainer_fold.eval_dataset).predictions
    p_oof = softmax(oof_logits, axis=-1)
    oof_proba_a_muril[va_idx] = p_oof

    f1_fold = f1_score(y_train_a_int[va_idx], np.argmax(p_oof, axis=-1),
                       average="macro")
    fold_f1s_a_muril.append(f1_fold)
    print(f"  Fold {fold+1} OOF F1: {f1_fold:.4f}")

    # Dev
    dev_logits = trainer_fold.predict(
        make_sub_dataset(dev_a, "Comment_clean", MAX_LEN)
    ).predictions
    dev_probas_a_muril.append(softmax(dev_logits, axis=-1))

    # Test
    test_logits = trainer_fold.predict(
        make_sub_dataset(test_a, "Comment_clean", MAX_LEN)
    ).predictions
    test_probas_a_muril.append(softmax(test_logits, axis=-1))

    # Teardown
    del trainer_fold, oof_logits, dev_logits, test_logits
    for _ in range(3): gc.collect()
    torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    if os.path.exists(f"./out_a_muril_fold{fold}"):
        shutil.rmtree(f"./out_a_muril_fold{fold}", ignore_errors=True)

    free_gb, _ = [x / 1e9 for x in torch.cuda.mem_get_info()]
    print(f"  GPU free after fold {fold+1}: {free_gb:.2f} GB")

    # Auto-save
    with open("/kaggle/working/saved_session/task_a_muril_partial.pkl", "wb") as f:
        pickle.dump({
            "oof_proba_a_muril": oof_proba_a_muril,
            "dev_probas_a_muril": dev_probas_a_muril,
            "test_probas_a_muril": test_probas_a_muril,
            "fold_f1s_a_muril": fold_f1s_a_muril,
        }, f)
    print(f"  ✓ Auto-saved fold {fold+1}")

# Aggregate
DEV_PROBA_A_MURIL  = np.mean(dev_probas_a_muril,  axis=0)
TEST_PROBA_A_MURIL = np.mean(test_probas_a_muril, axis=0)

oof_pred = np.argmax(oof_proba_a_muril, axis=-1)
oof_f1_a_muril = f1_score(y_train_a_int, oof_pred, average="macro")
dev_f1_a_muril = f1_score(y_dev_a_int, np.argmax(DEV_PROBA_A_MURIL, axis=-1),
                          average="macro")

print(f"\n{'='*60}")
print(f"Task A — MuRIL 5-Fold CV Results")
print(f"{'='*60}")
print(f"Per-fold OOF F1s: {[f'{x:.4f}' for x in fold_f1s_a_muril]}")
print(f"Mean ± std: {np.mean(fold_f1s_a_muril):.4f} ± {np.std(fold_f1s_a_muril):.4f}")
print(f"\nOOF CV F1: {oof_f1_a_muril:.4f}")
print(f"Dev F1:    {dev_f1_a_muril:.4f}")
print(f"{'='*60}")
print(classification_report(y_dev_a_int, np.argmax(DEV_PROBA_A_MURIL, axis=-1),
                            target_names=LABELS_A, digits=3))

# Save
with open("/kaggle/working/saved_session/task_a_muril.pkl", "wb") as f:
    pickle.dump({
        "oof_proba_a_muril":  oof_proba_a_muril,
        "DEV_PROBA_A_MURIL":  DEV_PROBA_A_MURIL,
        "TEST_PROBA_A_MURIL": TEST_PROBA_A_MURIL,
        "fold_f1s_a_muril":   fold_f1s_a_muril,
        "oof_f1_a_muril":     oof_f1_a_muril,
        "dev_f1_a_muril":     dev_f1_a_muril,
    }, f)

print(f"\n✓ Saved to /kaggle/working/saved_session/task_a_muril.pkl")
print(f"✓ Cell 11 complete")

GPU free at start: 15.34 GB

Fold 1/5  train=5156  val=1290
GPU free before fold: 15.34 GB


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  953MB            

... (54 outputs left out)

  Fold 5 OOF F1: 0.7905


Map:   0%|          | 0/806 [00:00<?, ? examples/s]

Map:   0%|          | 0/806 [00:00<?, ? examples/s]

  GPU free after fold 5: 15.34 GB
  ✓ Auto-saved fold 5

Task A — MuRIL 5-Fold CV Results
Per-fold OOF F1s: ['0.8163', '0.7747', '0.7882', '0.7834', '0.7905']
Mean ± std: 0.7906 ± 0.0139

OOF CV F1: 0.7907
Dev F1:    0.8064
              precision    recall  f1-score   support

        Hate      0.785     0.833     0.808       395
    Non-Hate      0.829     0.781     0.805       411

    accuracy                          0.806       806
   macro avg      0.807     0.807     0.806       806
weighted avg      0.808     0.806     0.806       806


✓ Saved to /kaggle/working/saved_session/task_a_muril.pkl
✓ Cell 11 complete


In [15]:
# ============================================================
# Cell 12: Task B — MuRIL 5-fold CV
# ============================================================
from sklearn.model_selection import StratifiedKFold
import gc, pickle, os

gc.collect(); torch.cuda.empty_cache()
free_gb, _ = [x / 1e9 for x in torch.cuda.mem_get_info()]
print(f"GPU free at start: {free_gb:.2f} GB")

MURIL = "google/muril-base-cased"
skf_b = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_proba_b_muril = np.zeros((len(train_b), len(LABELS_B)), dtype=np.float32)
dev_probas_b_muril  = []
test_probas_b_muril = []
fold_f1s_b_muril    = []

for fold, (tr_idx, va_idx) in enumerate(skf_b.split(train_b, y_train_b_int)):
    print(f"\n{'='*60}")
    print(f"Fold {fold+1}/{N_FOLDS}  train={len(tr_idx)}  val={len(va_idx)}")
    free_gb, _ = [x / 1e9 for x in torch.cuda.mem_get_info()]
    print(f"GPU free before fold: {free_gb:.2f} GB")
    print(f"{'='*60}")

    fold_train = train_b.iloc[tr_idx].reset_index(drop=True)
    fold_val   = train_b.iloc[va_idx].reset_index(drop=True)

    for _ in range(2): gc.collect()
    torch.cuda.empty_cache(); torch.cuda.synchronize()

    trainer_fold, _, _ = make_rdrop_trainer(
        MURIL, fold_train, fold_val,
        text_col="Comment_clean", label_col="Hate Category",
        labels=LABELS_B, l2id=L2ID_B,
        epochs=5,
        bs=8, grad_accum=2,
        lr=2e-5,
        weights=W_B,
        out_dir=f"./out_b_muril_fold{fold}",
        max_len=MAX_LEN,
        gradient_checkpointing=True,
        optim="adamw_torch",
        fp16=True,
        rdrop_alpha=0.5, label_smoothing=0.1,
        save_strategy="no",
    )
    trainer_fold.train()

    # OOF
    oof_logits = trainer_fold.predict(trainer_fold.eval_dataset).predictions
    p_oof = softmax(oof_logits, axis=-1)
    oof_proba_b_muril[va_idx] = p_oof

    f1_fold = f1_score(y_train_b_int[va_idx], np.argmax(p_oof, axis=-1),
                       average="macro")
    fold_f1s_b_muril.append(f1_fold)
    print(f"  Fold {fold+1} OOF F1: {f1_fold:.4f}")

    # Dev
    dev_logits = trainer_fold.predict(
        make_sub_dataset(dev_b, "Comment_clean", MAX_LEN)
    ).predictions
    dev_probas_b_muril.append(softmax(dev_logits, axis=-1))

    # Test
    test_logits = trainer_fold.predict(
        make_sub_dataset(test_b, "Comment_clean", MAX_LEN)
    ).predictions
    test_probas_b_muril.append(softmax(test_logits, axis=-1))

    # Teardown
    del trainer_fold, oof_logits, dev_logits, test_logits
    for _ in range(3): gc.collect()
    torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    if os.path.exists(f"./out_b_muril_fold{fold}"):
        shutil.rmtree(f"./out_b_muril_fold{fold}", ignore_errors=True)

    free_gb, _ = [x / 1e9 for x in torch.cuda.mem_get_info()]
    print(f"  GPU free after fold {fold+1}: {free_gb:.2f} GB")

    # Auto-save
    with open("/kaggle/working/saved_session/task_b_muril_partial.pkl", "wb") as f:
        pickle.dump({
            "oof_proba_b_muril": oof_proba_b_muril,
            "dev_probas_b_muril": dev_probas_b_muril,
            "test_probas_b_muril": test_probas_b_muril,
            "fold_f1s_b_muril": fold_f1s_b_muril,
        }, f)
    print(f"  ✓ Auto-saved fold {fold+1}")

# Aggregate
DEV_PROBA_B_MURIL  = np.mean(dev_probas_b_muril,  axis=0)
TEST_PROBA_B_MURIL = np.mean(test_probas_b_muril, axis=0)

oof_pred = np.argmax(oof_proba_b_muril, axis=-1)
oof_f1_b_muril = f1_score(y_train_b_int, oof_pred, average="macro")
dev_f1_b_muril = f1_score(y_dev_b_int, np.argmax(DEV_PROBA_B_MURIL, axis=-1),
                          average="macro")

print(f"\n{'='*60}")
print(f"Task B — MuRIL 5-Fold CV Results")
print(f"{'='*60}")
print(f"Per-fold OOF F1s: {[f'{x:.4f}' for x in fold_f1s_b_muril]}")
print(f"Mean ± std: {np.mean(fold_f1s_b_muril):.4f} ± {np.std(fold_f1s_b_muril):.4f}")
print(f"\nOOF CV F1: {oof_f1_b_muril:.4f}")
print(f"Dev F1:    {dev_f1_b_muril:.4f}")
print(f"{'='*60}")
print(classification_report(y_dev_b_int, np.argmax(DEV_PROBA_B_MURIL, axis=-1),
                            target_names=LABELS_B, digits=3))

# Save
with open("/kaggle/working/saved_session/task_b_muril.pkl", "wb") as f:
    pickle.dump({
        "oof_proba_b_muril":  oof_proba_b_muril,
        "DEV_PROBA_B_MURIL":  DEV_PROBA_B_MURIL,
        "TEST_PROBA_B_MURIL": TEST_PROBA_B_MURIL,
        "fold_f1s_b_muril":   fold_f1s_b_muril,
        "oof_f1_b_muril":     oof_f1_b_muril,
        "dev_f1_b_muril":     dev_f1_b_muril,
    }, f)

print(f"\n✓ Saved to /kaggle/working/saved_session/task_b_muril.pkl")
print(f"✓ Cell 12 complete")

GPU free at start: 15.34 GB

Fold 1/5  train=2527  val=632
GPU free before fold: 15.34 GB


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

Map:   0%|          | 0/2527 [00:00<?, ? examples/s]

Map:   0%|          | 0/2527 [00:00<?, ? examples/s]

Map:   0%|          | 0/632 [00:00<?, ? examples/s]

... (47 outputs left out)

Map:   0%|          | 0/395 [00:00<?, ? examples/s]

Map:   0%|          | 0/396 [00:00<?, ? examples/s]

  GPU free after fold 5: 15.34 GB
  ✓ Auto-saved fold 5

Task B — MuRIL 5-Fold CV Results
Per-fold OOF F1s: ['0.4745', '0.4809', '0.4167', '0.4892', '0.4750']
Mean ± std: 0.4672 ± 0.0258

OOF CV F1: 0.4684
Dev F1:    0.4944
               precision    recall  f1-score   support

       Gender      0.636     0.924     0.753       170
Geo-political      0.581     0.783     0.667        23
       Others      0.000     0.000     0.000        56
    Political      0.800     0.857     0.828        70
     Religion      0.780     0.667     0.719        48
     Violence      0.000     0.000     0.000        28

     accuracy                          0.676       395
    macro avg      0.466     0.538     0.494       395
 weighted avg      0.544     0.676     0.597       395


✓ Saved to /kaggle/working/saved_session/task_b_muril.pkl
✓ Cell 12 complete


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [17]:
# ============================================================
# Cell 13: FINAL SUBMISSION
#   Task A: 3-model ensemble (classical + XLM-R + MuRIL)
#   Task B: classical only (best single — MuRIL broke)
# ============================================================
import pickle, numpy as np, pandas as pd, itertools, os
from sklearn.metrics import f1_score, classification_report

# ============================================================
# TASK A — 3-model ensemble
# ============================================================
print("=" * 70)
print("TASK A — 3-Model Ensemble")
print("=" * 70)

# Load MuRIL Task A (already computed successfully in Cell 11)
with open("/kaggle/working/saved_session/task_a_muril.pkl", "rb") as f:
    ma = pickle.load(f)

DEV_PROBA_A_MURIL  = ma["DEV_PROBA_A_MURIL"]
TEST_PROBA_A_MURIL = ma["TEST_PROBA_A_MURIL"]

y_dev_a_int = np.array([L2ID_A[x] for x in dev_a["Label"].values])

members_dev_a = {
    "classical": stack_proba_dev_a,
    "xlmr":      DEV_PROBA_A_XLMR,
    "muril":     DEV_PROBA_A_MURIL,
}
members_test_a = {
    "classical": stack_proba_test_a,
    "xlmr":      TEST_PROBA_A_XLMR,
    "muril":     TEST_PROBA_A_MURIL,
}

print("Individual members on dev_a:")
for name, p in members_dev_a.items():
    f1 = f1_score(y_dev_a_int, np.argmax(p, axis=-1), average="macro")
    print(f"  {name:12s} F1={f1:.4f}")

# 3-model grid search
best_f1_a, best_w_a = 0, None
for w_c, w_x, w_m in itertools.product(np.arange(0, 1.01, 0.05), repeat=3):
    if abs(w_c + w_x + w_m - 1.0) > 0.01: continue
    ens = w_c*members_dev_a["classical"] + w_x*members_dev_a["xlmr"] + w_m*members_dev_a["muril"]
    f1 = f1_score(y_dev_a_int, np.argmax(ens, axis=-1), average="macro")
    if f1 > best_f1_a:
        best_f1_a, best_w_a = f1, (w_c, w_x, w_m)

w_c_a, w_x_a, w_m_a = best_w_a
print(f"\nBest weights: classical={w_c_a:.2f}  xlmr={w_x_a:.2f}  muril={w_m_a:.2f}")
print(f"Task A ensemble dev F1: {best_f1_a:.4f}")

# Threshold tuning
ens_dev_a = (w_c_a*members_dev_a["classical"] + w_x_a*members_dev_a["xlmr"]
           + w_m_a*members_dev_a["muril"])
best_thresh_a, best_f1_a_th = 0.5, best_f1_a
for t in np.arange(0.35, 0.66, 0.005):
    preds_t = (ens_dev_a[:, L2ID_A["Non-Hate"]] >= t).astype(int)
    f1 = f1_score(y_dev_a_int, preds_t, average="macro")
    if f1 > best_f1_a_th:
        best_f1_a_th, best_thresh_a = f1, t
print(f"After threshold tuning (t={best_thresh_a:.3f}): {best_f1_a_th:.4f}")

# Task A test predictions
ens_test_a = (w_c_a*members_test_a["classical"] + w_x_a*members_test_a["xlmr"]
            + w_m_a*members_test_a["muril"])
preds_a_test = (ens_test_a[:, L2ID_A["Non-Hate"]] >= best_thresh_a).astype(int)
submission_a = pd.DataFrame({
    "id":    test_a["id"].values,
    "label": [ID2L_A[i] for i in preds_a_test],
})

print(f"\nTask A submission distribution:")
print(submission_a["label"].value_counts().to_string())


# ============================================================
# TASK B — Classical only (best single model)
# ============================================================
print("\n" + "=" * 70)
print("TASK B — Classical Only")
print("=" * 70)

y_dev_b_int = np.array([L2ID_B[x] for x in dev_b["Hate Category"].values])

# Classical alone is best (dev 0.6648)
f1_dev_b_classical = f1_score(y_dev_b_int, np.argmax(stack_proba_dev_b, axis=-1),
                              average="macro")
print(f"Classical alone dev F1: {f1_dev_b_classical:.4f}")

# Check if XLM-R adds anything (with optimized weight)
best_f1_b, best_w_b = f1_dev_b_classical, (1.0, 0.0)  # default: classical only
for w_c in np.arange(0.5, 1.01, 0.05):
    w_x = 1.0 - w_c
    ens = w_c * stack_proba_dev_b + w_x * DEV_PROBA_B_XLMR
    f1 = f1_score(y_dev_b_int, np.argmax(ens, axis=-1), average="macro")
    if f1 > best_f1_b:
        best_f1_b, best_w_b = f1, (w_c, w_x)

w_c_b, w_x_b = best_w_b
print(f"\nBest Task B weights: classical={w_c_b:.2f}  xlmr={w_x_b:.2f}")
print(f"Task B ensemble dev F1: {best_f1_b:.4f}")

# Task B test predictions
ens_test_b = w_c_b * stack_proba_test_b + w_x_b * TEST_PROBA_B_XLMR
preds_b_test = np.argmax(ens_test_b, axis=-1)
submission_b = pd.DataFrame({
    "id":    test_b["id"].values,
    "label": [ID2L_B[i] for i in preds_b_test],
})

print(f"\nTask B submission distribution:")
print(submission_b["label"].value_counts().to_string())


# ============================================================
# SAVE SUBMISSIONS
# ============================================================
print("\n" + "=" * 70)
print("SAVING SUBMISSIONS")
print("=" * 70)

out_dir = "/kaggle/working/submissions"
os.makedirs(out_dir, exist_ok=True)

submission_a.to_csv(f"{out_dir}/predictions_task_a.csv", index=False)
submission_b.to_csv(f"{out_dir}/predictions_task_b.csv", index=False)

# Also save as v2 for comparison
submission_a.to_csv(f"{out_dir}/predictions_task_a_v2.csv", index=False)
submission_b.to_csv(f"{out_dir}/predictions_task_b_v2.csv", index=False)

print(f"✓ Saved CSVs to {out_dir}/")
for f in sorted(os.listdir(out_dir)):
    sz = os.path.getsize(f"{out_dir}/{f}")
    print(f"    {f}  ({sz/1024:.1f} KB)")

with open("/kaggle/working/saved_session/final_submission_v2.pkl", "wb") as f:
    pickle.dump({
        "best_f1_a": best_f1_a_th, "best_w_a": best_w_a, "best_thresh_a": best_thresh_a,
        "best_f1_b": best_f1_b, "best_w_b": best_w_b,
        "submission_a": submission_a, "submission_b": submission_b,
    }, f)

# ============================================================
# SUMMARY
# ============================================================
print("\n" + "=" * 70)
print("FINAL SUMMARY (v2 with MuRIL)")
print("=" * 70)
print(f"\nTask A — Dev F1: {best_f1_a_th:.4f}  (was 0.8213)")
print(f"  Weights: classical={w_c_a:.2f} xlmr={w_x_a:.2f} muril={w_m_a:.2f}  thresh={best_thresh_a:.3f}")
print(f"\nTask B — Dev F1: {best_f1_b:.4f}  (was 0.6648)")
print(f"  Weights: classical={w_c_b:.2f} xlmr={w_x_b:.2f}")

print(f"\n✓ Cell 13 complete")

TASK A — 3-Model Ensemble
Individual members on dev_a:
  classical    F1=0.8002
  xlmr         F1=0.8002
  muril        F1=0.8064

Best weights: classical=0.45  xlmr=0.25  muril=0.30
Task A ensemble dev F1: 0.8288
After threshold tuning (t=0.500): 0.8288

Task A submission distribution:
label
Hate        422
Non-Hate    384

TASK B — Classical Only
Classical alone dev F1: 0.6648

Best Task B weights: classical=1.00  xlmr=0.00
Task B ensemble dev F1: 0.6648

Task B submission distribution:
label
Gender           156
Political         67
Others            62
Religion          42
Violence          37
Geo-political     32

SAVING SUBMISSIONS
✓ Saved CSVs to /kaggle/working/submissions/
    predictions_task_a.csv  (9.2 KB)
    predictions_task_a_v2.csv  (9.2 KB)
    predictions_task_b.csv  (5.2 KB)
    predictions_task_b_v2.csv  (5.2 KB)
    sample_task_a_predictions.csv  (9.3 KB)
    sample_task_b_predictions.csv  (5.2 KB)

FINAL SUMMARY (v2 with MuRIL)

Task A — Dev F1: 0.8288  (was 0.821

In [20]:
# ============================================================
# Cell 15: Package FINAL submissions (v2) as ZIP
# ============================================================
import zipfile, os
import pandas as pd

out_dir = "/kaggle/working/submissions"

# ---- Load final v2 files ----
final_a = pd.read_csv(f"{out_dir}/predictions_task_a.csv")
final_b = pd.read_csv(f"{out_dir}/predictions_task_b.csv")

print("=" * 70)
print("FINAL SUBMISSION VERIFICATION")
print("=" * 70)

print(f"\nTask A:")
print(f"  Rows: {len(final_a)}  (expected 806)")
print(f"  Columns: {list(final_a.columns)}")
print(f"  Labels used: {sorted(final_a['label'].unique())}")
print(f"  Distribution:\n{final_a['label'].value_counts().to_string()}")

print(f"\nTask B:")
print(f"  Rows: {len(final_b)}  (expected 396)")
print(f"  Columns: {list(final_b.columns)}")
print(f"  Labels used: {sorted(final_b['label'].unique())}")
print(f"  Distribution:\n{final_b['label'].value_counts().to_string()}")

# Verify expected labels
expected_a = {"Hate", "Non-Hate"}
expected_b = {"Gender", "Geo-political", "Others", "Political", "Religion", "Violence"}
actual_a = set(final_a["label"].unique())
actual_b = set(final_b["label"].unique())

assert actual_a == expected_a, f"Task A label mismatch: {actual_a} vs {expected_a}"
assert actual_b == expected_b, f"Task B label mismatch: {actual_b} vs {expected_b}"
print("\n✓ Labels match expected format")

# Verify IDs
assert set(final_a["id"]) == set(test_a["id"]), "Task A ID mismatch!"
assert set(final_b["id"]) == set(test_b["id"]), "Task B ID mismatch!"
print("✓ IDs match test files")

# ---- Create download ZIPs ----
with zipfile.ZipFile(f"{out_dir}/FINAL_task_a.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write(f"{out_dir}/predictions_task_a.csv", "predictions_task_a.csv")

with zipfile.ZipFile(f"{out_dir}/FINAL_task_b.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write(f"{out_dir}/predictions_task_b.csv", "predictions_task_b.csv")

with zipfile.ZipFile(f"{out_dir}/FINAL_both.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write(f"{out_dir}/predictions_task_a.csv", "predictions_task_a.csv")
    z.write(f"{out_dir}/predictions_task_b.csv", "predictions_task_b.csv")

print(f"\n✓ Created 3 ZIP files in {out_dir}/")
for f in sorted(os.listdir(out_dir)):
    if f.endswith(".zip"):
        sz = os.path.getsize(f"{out_dir}/{f}")
        print(f"    {f}  ({sz/1024:.1f} KB)")

print(f"\n🎯 FINAL SUBMISSION READY")
print(f"  Task A: dev F1 = 0.8288")
print(f"  Task B: dev F1 = 0.6648")
print(f"  Files: FINAL_both.zip (or individual zips)")

FINAL SUBMISSION VERIFICATION

Task A:
  Rows: 806  (expected 806)
  Columns: ['id', 'label']
  Labels used: ['Hate', 'Non-Hate']
  Distribution:
label
Hate        422
Non-Hate    384

Task B:
  Rows: 396  (expected 396)
  Columns: ['id', 'label']
  Labels used: ['Gender', 'Geo-political', 'Others', 'Political', 'Religion', 'Violence']
  Distribution:
label
Gender           156
Political         67
Others            62
Religion          42
Violence          37
Geo-political     32

✓ Labels match expected format
✓ IDs match test files

✓ Created 3 ZIP files in /kaggle/working/submissions/
    FINAL_both.zip  (4.0 KB)
    FINAL_task_a.zip  (2.5 KB)
    FINAL_task_b.zip  (1.5 KB)

🎯 FINAL SUBMISSION READY
  Task A: dev F1 = 0.8288
  Task B: dev F1 = 0.6648
  Files: FINAL_both.zip (or individual zips)


In [21]:
# ============================================================
# Cell 16: Create form-ready CSV files with correct names
# ============================================================
import shutil, os, pandas as pd

out_dir = "/kaggle/working/submissions"

# Team name from form: "Israt Jahan Mou"
# Use simple version without spaces (safer for filenames)
team_slug = "IsratJahanMou"

# Read existing submissions
final_a = pd.read_csv(f"{out_dir}/predictions_task_a.csv")
final_b = pd.read_csv(f"{out_dir}/predictions_task_b.csv")

# Save with form-expected names
a_filename = f"{team_slug}_taskA.csv"
b_filename = f"{team_slug}_taskB.csv"

final_a.to_csv(f"{out_dir}/{a_filename}", index=False)
final_b.to_csv(f"{out_dir}/{b_filename}", index=False)

print(f"✓ Form-ready files created:")
print(f"  {out_dir}/{a_filename}")
print(f"  {out_dir}/{b_filename}")
print(f"\nFile details:")
print(f"  {a_filename}: {len(final_a)} rows, {os.path.getsize(f'{out_dir}/{a_filename}')/1024:.1f} KB")
print(f"  {b_filename}: {len(final_b)} rows, {os.path.getsize(f'{out_dir}/{b_filename}')/1024:.1f} KB")

# Also make individual zips (in case needed)
import zipfile
with zipfile.ZipFile(f"{out_dir}/{team_slug}_taskA.zip", "w") as z:
    z.write(f"{out_dir}/{a_filename}", a_filename)
with zipfile.ZipFile(f"{out_dir}/{team_slug}_taskB.zip", "w") as z:
    z.write(f"{out_dir}/{b_filename}", b_filename)

print(f"\nZips also created:")
print(f"  {team_slug}_taskA.zip")
print(f"  {team_slug}_taskB.zip")

✓ Form-ready files created:
  /kaggle/working/submissions/IsratJahanMou_taskA.csv
  /kaggle/working/submissions/IsratJahanMou_taskB.csv

File details:
  IsratJahanMou_taskA.csv: 806 rows, 9.2 KB
  IsratJahanMou_taskB.csv: 396 rows, 5.2 KB

Zips also created:
  IsratJahanMou_taskA.zip
  IsratJahanMou_taskB.zip
